# **Q-Learning with Keras**

### Set up the environment

In [1]:
import sys 
sys.setrecursionlimit(1500)
import gym
import numpy as np
env = gym.make('CartPole-v1')

np.random.seed(42)
env.action_space.seed(42)
env.observation_space.seed(42)

Gym has been unmaintained since 2022 and does not support NumPy 2.0 amongst other critical functionality.
Please upgrade to Gymnasium, the maintained drop-in replacement of Gym, or contact the authors of your software and request that they upgrade.
Users of this version of Gym should be able to simply replace 'import gym' with 'import gymnasium as gym' in the vast majority of cases.
See the migration guide at https://gymnasium.farama.org/introduction/migration_guide/ for additional information.


[42]

### Define the Q-Learning Model

In [2]:
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import Dense, Input
from tensorflow.keras.optimizers import Adam

# Define the model 
def build_model(state_size, action_size):
    model = Sequential()
    model.add(Input(shape=(state_size,)))
    model.add(Dense(24,activation='relu'))
    model.add(Dense(24,activation='relu'))
    model.add(Dense(action_size,activation='linear'))
    model.compile(loss='mse',optimizer=Adam(learning_rate=0.001))
    return model

In [5]:
import gym
env = gym.make('CartPole-v1')
state_size = env.observation_space.shape[0]
action_size = env.action_space.n
model = build_model(state_size,action_size)

### Implement the Q-Learning Algorithm

In [7]:
import random
import numpy as np
from collections import deque
import tensorflow as tf

epsilon = 1.0 # starting with a high exploration rate
epsilon_min = 0.01 # Minimun exploration rate
epsilon_decay = 0.99 # Faster decay rate for epsilon after each episode

# Replay memory
memory = deque(maxlen=2000)

def remember(state,action,reward,next_state,done):
    memory.append((state,action,reward,next_state,done))

def replay(batch_size=64):
    if len(memory) < batch_size:
        return # skip replay if there's not enough experience
    
    minibatch = random.sample(memory, batch_size)
    # Extract information for batch processing 
    states = np.vstack([x[0] for x in minibatch])
    actions = np.array([x[1] for x in minibatch])
    rewards = np.array([x[2] for x in minibatch],dtype=np.float32)
    next_states = np.vstack([x[3] for x in minibatch])
    dones = np.array([x[4] for x in minibatch],dtype=np.float32)

    # Predict Q-values for the next state in batch
    q_next = model.predict(next_states,verbose=0)
    # Predict Q-values for the next state in batch
    q_target = model.predict(states,verbose=0)

    #Vectorized update of target values
    for i in range(batch_size):
        target = rewards[i]
        if not dones[i]:
            target += 0.95 * np.amax(q_next[i]) # Update Q value with discounted future reward
        q_target[i][actions[i]] = target # Update only the taken action's Q value

    # train the model with the update target in batch
    model.fit(states,q_target,epochs=1, verbose=0)

    # Reduce exploration rate (epsilon) after each training step
    global epsilon
    if epsilon > epsilon_min:
        epsilon *= epsilon_decay

def act(state):
    if np.random.rand() <= epsilon:
        return random.randrange(action_size) # Exploit: choose a random action
    act_values = model.predict(state) # Exploit: predict action based on the state
    return np.argmax(act_values[0]) # return the action with the highest Q-value

# Define the number of episodes you want to train the model
episodes = 10
train_frequency = 5

for e in range(episodes):
    state, _ = env.reset() # Unpack the tuple returned by env.reset()
    state = np.reshape(state, [1,state_size])
    for time in range (200): # limit to 200 time step per episode
        action = act(state)
        next_state, reward, terminated, truncated , _ = env.step(action)
        done = terminated or truncated
        reward = reward if not done else -10
        next_state = np.reshape(next_state,[1, state_size])
        remember(state,action,reward,next_state,done) # Store experience
        state = next_state

        if done:
            print(f"episode: {e+1}/{episodes}, score: {time} , e: {epsilon:.2}")
            break

        # train the model every train_freqency step
        if time % train_frequency == 0:
            replay(batch_size=64) # call replay with larger batch size for efficiency

    env.close()

episode: 1/10, score: 15 , e: 1.0
episode: 2/10, score: 19 , e: 1.0
episode: 3/10, score: 10 , e: 1.0
episode: 4/10, score: 43 , e: 0.95
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 89ms/step
episode: 5/10, score: 23 , e: 0.9
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 50ms/step
episode: 6/10, score: 7 , e: 0.89
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 47ms/step
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 46ms/step
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 47ms/step
episode: 7/10, score: 22 , e: 0.84
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 47ms/step
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 47ms/step
episode: 8/10, score: 9 , e: 0.83
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 46ms/step
episode: 9/10, score: 15 , e: 0.8
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 46ms/step
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 46ms/step
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 47ms/step
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 47ms/step
episode: 10/10, score: 12 , e: 0.78


### Evaluate the performance

In [8]:
for e in range(10):
    state,_ = env.reset() # Unpack the state from the tuple
    state = np.reshape(state,[1,state_size]) # Reshape the state correctly
    for time in range(500):
        env.render()
        action = np.argmax(model.predict(state)[0])
        next_state, reward, terminated, truncated, _ = env.step(action) # Unpack the five return values
        done = terminated or truncated # Check if the episode is done
        next_state = np.reshape(next_state, [1,state_size])
        state = next_state
        if done:
            print(f'episode: {e+1}/10, score: {time}')
            break
env.close()

1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 68ms/step


C:\Users\ilyass\AppData\Roaming\Python\Python313\site-packages\gym\envs\classic_control\cartpole.py:211: UserWarning: WARN: You are calling render method without specifying any render mode. You can specify the render_mode at initialization, e.g. gym("CartPole-v1", render_mode="rgb_array")
  gym.logger.warn(


1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 66ms/step
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 56ms/step
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 56ms/step
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 61ms/step
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 68ms/step
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 55ms/step
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 61ms/step
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 66ms/step
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 68ms/step
episode: 1/10, score: 9
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 57ms/step
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 61ms/step
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 58ms/step
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 58ms/step
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 59ms/step
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 55ms/step
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 56ms/step
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 53ms/step
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 55ms/step
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 53ms/step
episode: 2/10, score: 9
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 60ms/step
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 52ms/step
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 51ms/step
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 51ms/step
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 53ms/step
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 52ms/step
1/